# FunctorFlow PTB Model Comparison

This notebook compares three language-model families on the local Penn
Treebank corpus using one shared `FunctorFlow/` data and training path:

- a baseline causal Transformer
- a GT-style topology-aware model
- the FunctorFlow-backed KET model


In [ ]:
from pathlib import Path
import sys

def _bootstrap_functorflow() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        package_root = candidate / "FunctorFlow"
        if (package_root / "__init__.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError(
        "Could not find the FunctorFlow package. Run this notebook from the repo "
        "root or from FunctorFlow/notebooks."
    )

REPO_ROOT = _bootstrap_functorflow()
print(f"FunctorFlow repo root: {REPO_ROOT}")


In [ ]:
from FunctorFlow.ket_lm import pick_device
from FunctorFlow.lm_compare import LMComparisonConfig, compare_language_models

device = pick_device("cpu")
config = LMComparisonConfig(
    steps=2,
    block_size=32,
    batch_size=4,
    lr=2e-3,
    model_profile="smoke",
    train_tokens=2048,
    valid_tokens=512,
    test_tokens=512,
    seed=0,
)
comparison = compare_language_models("ptb", comparison_config=config, device=device)
comparison["corpus"], comparison["model_profile"], comparison["train_tokens"], comparison["valid_tokens"]


## Validation Perplexity

This is still a smoke-scale run, but it gives the tutorial a real executable
comparison surface for Transformer, GT, and KET on one benchmark.


In [ ]:
{
    model_name: round(model_result["valid_ppl"], 2)
    for model_name, model_result in comparison["models"].items()
}


In [ ]:
{
    model_name: model_result["history"]["train_loss"]
    for model_name, model_result in comparison["models"].items()
}
